In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
customer = spark.read.table('main.silver_task.customers')
order = spark.read.table('main.silver_task.orders')
payment = spark.read.table('main.silver_task.payments')
product = spark.read.table('main.silver_task.products')
category = spark.read.table('main.silver_task.categories')

In [0]:
gold1_df = customer.join(order,'customer_id').groupBy('customer_id')\
    .agg(count('*').alias('total_orders'),sum('amount').alias('total_spent')\
        ,avg('amount').alias('avg_amount'))

gold1_df = gold1_df.withColumn('customer_tier',
        when(col('total_spent')>=1000,'platinum')
        .when(col('total_spent')>=600,'gold')
        .when(col('total_spent')>=400,'silver')
        .otherwise('bronze'))

gold1_df.write.mode('overwrite').saveAsTable('main.gold_task.customers_order_summary')

In [0]:
gold2_df = order.join(product,'product_id').join(category,'category_id')\
    .groupBy('product_id','category_name','product_name').agg(count('*').alias('total_orders')\
        ,sum('amount').alias('total_revenue'),avg('amount').alias('avg_order_amount'))
    
w = Window.partitionBy('category_name').orderBy(col('total_revenue').desc())
gold2_df = gold2_df.withColumn('category_rank',rank().over(w))

gold2_df.write.mode('overwrite').saveAsTable('main.gold_task.product_sale_summary')

In [0]:
gold3_df = customer.alias('c').join(order.alias('o'),'customer_id').join(payment.alias('p'),'order_id')\
    .groupBy('c.customer_id','c.customer_name').agg(count('*').alias('total_payments')\
        ,sum('p.amount').alias('total_paid'),avg('p.amount').alias('avg_payment_amount'))
    
gold3_df = gold3_df.withColumn(
    "payment_category",
    when(col("total_paid") >= 800, "HIGH VALUE")
    .when(col("total_paid") >= 500, "MEDIUM VALUE")
    .otherwise("LOW VALUE")
)

gold3_df.write.mode("overwrite").saveAsTable("main.gold_task.customer_payment_summary")